In [1]:
import time

import pandas as pd
import requests

api_url = "https://collectionapi.metmuseum.org/public/collection/v1"
my_headers = {
    "User-Agent": "hse-hw1-met (https://vk.ru/id488676249)"
}
timeout = (10, 60)  # (соединение, чтение ответа)
pause = 2.0

departments = {
    11: "European Paintings",
    10: "Egyptian Art",
    4: "Arms and Armor",
}

In [2]:
def fetch_json(endpoint: str, params: dict | None = None) -> dict:
    """Отправляет GET-запрос к API Met и возвращает ответ в виде словаря."""

    response = requests.get(
        endpoint,
        params=params,
        headers=my_headers,
        timeout=timeout,
    )
    response.raise_for_status()  # код 4xx или 5xx — остановиться с ошибкой

    time.sleep(pause)  # пауза, чтобы не перегружать сервер

    return response.json()

In [3]:
ids_per_department = 100
all_object_ids = []

for dept_id, dept_name in departments.items():
    print(f"Сбор ID для отдела: {dept_name} (ID {dept_id})")

    data = fetch_json(f"{api_url}/objects", params={"departmentIds": dept_id})

    if not data.get("objectIDs"):
        print(f"  Не удалось получить ID для отдела {dept_name}")
        continue

    all_ids = data["objectIDs"]
    ids = all_ids[:ids_per_department]
    print(f"  Получено ID: {len(ids)} (из {len(all_ids)} доступных)")

    for oid in ids:
        all_object_ids.append(oid)

print(f"\nВсего ID для сбора метаданных: {len(all_object_ids)}")

Сбор ID для отдела: European Paintings (ID 11)
  Получено ID: 100 (из 2650 доступных)
Сбор ID для отдела: Egyptian Art (ID 10)
  Получено ID: 100 (из 28159 доступных)
Сбор ID для отдела: Arms and Armor (ID 4)
  Получено ID: 100 (из 13779 доступных)

Всего ID для сбора метаданных: 300


In [4]:
records = []

for i, oid in enumerate(all_object_ids):
    try:
        obj = fetch_json(f"{api_url}/objects/{oid}")
    except requests.exceptions.RequestException as e:
        print(f"  Пропущен объект {oid}: {e}")
        time.sleep(5)
        continue

    records.append({
        "objectID": obj.get("objectID"),
        "title": obj.get("title"),
        "artistDisplayName": obj.get("artistDisplayName", ""),
        "department": obj.get("department"),
        "classification": obj.get("classification", ""),
        "culture": obj.get("culture", ""),
        "objectDate": obj.get("objectDate", ""),
        "medium": obj.get("medium", ""),
        "isPublicDomain": obj.get("isPublicDomain"),
    })

    if (i + 1) % 50 == 0:
        print(f"Обработано: {i + 1} / {len(all_object_ids)}")


Обработано: 50 / 300
Обработано: 100 / 300
  Пропущен объект 543873: HTTPSConnectionPool(host='collectionapi.metmuseum.org', port=443): Read timed out. (read timeout=10)
Обработано: 150 / 300
Обработано: 200 / 300
  Пропущен объект 22380: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
Обработано: 250 / 300
Обработано: 300 / 300

Собрано записей: 298


,objectID,title,artistDisplayName,department,classification,culture,objectDate,medium,isPublicDomain
0,436582,Head of a Young Woman,Jean-Baptiste Greuze,European Paintings,Paintings,,possibly 1780s,Oil on canvas,True
1,436583,Study of a Woman’s Head,Jean-Baptiste Greuze,European Paintings,Paintings,,ca. 1780,Oil on wood,True
2,436584,Madame Jean-Baptiste Nicolet (Anne Antoinette ...,Jean-Baptiste Greuze,European Paintings,Paintings,,late 1780s,Oil on wood,True
3,436586,Princess Varvara Nikolaevna Gagarina (1762–1802),Jean-Baptiste Greuze,European Paintings,Paintings,,ca. 1780–82,Oil on canvas,True
4,436587,Jean Jacques Caffiéri (1725–1792),Jean-Baptiste Greuze,European Paintings,Paintings,,ca. 1765,Oil on canvas,True


In [8]:
df = pd.DataFrame(records)
print(f"\nСобрано записей: {len(df)}")
df


Собрано записей: 298


,objectID,title,artistDisplayName,department,classification,culture,objectDate,medium,isPublicDomain
0,436582,Head of a Young Woman,Jean-Baptiste Greuze,European Paintings,Paintings,,possibly 1780s,Oil on canvas,True
1,436583,Study of a Woman’s Head,Jean-Baptiste Greuze,European Paintings,Paintings,,ca. 1780,Oil on wood,True
2,436584,Madame Jean-Baptiste Nicolet (Anne Antoinette ...,Jean-Baptiste Greuze,European Paintings,Paintings,,late 1780s,Oil on wood,True
3,436586,Princess Varvara Nikolaevna Gagarina (1762–1802),Jean-Baptiste Greuze,European Paintings,Paintings,,ca. 1780–82,Oil on canvas,True
4,436587,Jean Jacques Caffiéri (1725–1792),Jean-Baptiste Greuze,European Paintings,Paintings,,ca. 1765,Oil on canvas,True
...,...,...,...,...,...,...,...,...,...
293,22453,Harness Ornament,,Arms and Armor,Equestrian Equipment,Japanese,4th–7th century,"Iron, gold",True
294,22454,Harness Ornament,,Arms and Armor,Equestrian Equipment,Japanese,4th–7th century,"Iron, gold",True
295,22455,Spearhead,,Arms and Armor,Shafted Weapons,possibly Japanese,unknown,Bronze,True
296,22457,Helmet (Shokakutsuki Kabuto) in the Style of c...,,Arms and Armor,Helmets,Japanese,ca. 1900,Iron,True


In [10]:
df.to_csv("data.csv", index=False, encoding="utf-8")
print("Сохранено в data.csv")

Сохранено в data.csv
